# Inference: U-Net Burn Detection

Run trained model on test set to generate predictions for evaluation.

## Setup

In [ ]:
import os
if not os.path.exists('RETINNA'):
    !git clone https://github.com/scerruti/RETINNA.git

import sys
sys.path.insert(0, '/content/RETINNA' if '/content' in os.getcwd() else 'RETINNA')

In [ ]:
%pip install -q torchgeo torch torchvision matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.unet import UNet
from src.dataset import get_dataloaders
from src.device_utils import get_device, move_to_device

device = get_device()

## Load Data and Model

In [ ]:
# Setup Google Drive caching if on Colab
root = None
try:
    from src.colab_utils import setup_cabuaur_cached
    root = setup_cabuaur_cached()
    print("✓ Google Drive caching enabled")
except (ImportError, RuntimeError):
    print("Using default TorchGeo cache")

# Load dataloaders (test set only)
print("\nLoading test set...")
dataloaders = get_dataloaders(
    batch_size=4,
    num_workers=0,
    normalize=True,
    root=root
)

test_loader = dataloaders['test']
print(f"Test samples: {len(dataloaders['datasets']['test'])}")


In [ ]:
# Load trained model
checkpoint_path = Path('checkpoints_notebook/best_model.pth')
if not checkpoint_path.exists():
    print(f"Error: Checkpoint not found at {checkpoint_path}")
    print("Make sure to run 03_training.ipynb first and save the model.")
else:
    print(f"Loading checkpoint: {checkpoint_path}")
    model = UNet(in_channels=24, out_channels=2).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    print(f"✓ Model loaded: {model.get_parameter_count():,} parameters")

## Run Inference

In [ ]:
# Store predictions and ground truth
all_predictions = []
all_targets = []
all_images = []

print("Running inference on test set...")
with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        batch = move_to_device(batch, device)
        images = batch['image']
        masks = batch['mask']

        # Flatten timesteps
        B, T, C, H, W = images.shape
        images_flat = images.view(B, T * C, H, W)

        # Forward pass
        outputs = model(images_flat)  # [B, 2, H, W]
        probs = torch.softmax(outputs, dim=1)  # Probabilities
        burned_prob = probs[:, 1]  # Probability of burned class [B, H, W]

        # Store on CPU
        all_predictions.append(burned_prob.cpu())
        all_targets.append(masks.squeeze(1).cpu())
        all_images.append(images[:, 0].cpu())  # Store pre-fire image

        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {batch_idx + 1}/{len(test_loader)} batches")

# Concatenate all predictions
predictions = torch.cat(all_predictions, dim=0)
targets = torch.cat(all_targets, dim=0)
images = torch.cat(all_images, dim=0)

print(f"\n✓ Inference complete")
print(f"  Predictions shape: {predictions.shape}")
print(f"  Targets shape: {targets.shape}")
print(f"  Images shape: {images.shape}")

# Save predictions for analysis
output_dir = Path('inference_results')
output_dir.mkdir(exist_ok=True)
torch.save({'predictions': predictions, 'targets': targets, 'images': images}, output_dir / 'predictions.pt')
print(f"\n✓ Saved predictions to {output_dir}/predictions.pt")

## Sample Predictions

In [ ]:
# Show a few sample predictions
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Sample Predictions: Pre-Fire Image | Ground Truth | Prediction | Binary (T=0.5)', fontsize=12)

for i in range(3):
    # Pre-fire image (SWIR band for visualization)
    ax = axes[i, 0]
    img = images[i, 10].numpy()  # SWIR band (index 10)
    ax.imshow(img, cmap='gray')
    ax.set_title('Pre-Fire')
    ax.axis('off')

    # Ground truth
    ax = axes[i, 1]
    ax.imshow(targets[i].numpy(), cmap='RdYlGn_r', vmin=0, vmax=1)
    ax.set_title('Ground Truth')
    ax.axis('off')

    # Prediction probability
    ax = axes[i, 2]
    ax.imshow(predictions[i].numpy(), cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_title('Prediction (Prob)')
    ax.axis('off')

    # Binary prediction (threshold 0.5)
    ax = axes[i, 3]
    pred_binary = (predictions[i] > 0.5).float()
    ax.imshow(pred_binary.numpy(), cmap='RdYlGn_r', vmin=0, vmax=1)
    ax.set_title('Binary (T=0.5)')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\nSample predictions saved. Use 05_analysis.ipynb for detailed metrics.")
